# HyperSIGMA — Binary Change Detection on MUMUCD

Adaptation du pipeline de segmentation HyperSIGMA pour la détection de changement binaire
sur les données hyperspectrales PRISMA du dataset MUMUCD.

## 1. Configuration

Choisir la ville et les hyperparamètres du pipeline.

In [ ]:
# ============================================================
# CONFIGURATION — modifier ici
# ============================================================

# Nom de la ville (dossier dans data/mumucd/)
CITY = "paris"

# Chemins
DATA_ROOT = "data/mumucd"
WEIGHT_DIR = "weight"
MODEL_DIR = "HyperSIGMA/ImageClassification"

# Hyperparamètres
PCA_COMPONENTS = 30       # Réduction spectrale (avant+après concaténés = 460 bandes → 30)
IMG_SIZE = 128            # Taille des patches pour le modèle
PATCH_SIZE = 8            # Patch size du ViT
OVERLAP_SIZE = 32         # Overlap entre patches lors de l'extraction
BATCH_SIZE = 4            # Batch size pour l'entraînement
NUM_EPOCHS = 50           # Nombre d'epochs
LR = 6e-5                 # Learning rate
WEIGHT_DECAY = 0.05
TRAIN_RATIO = 0.05        # Ratio de pixels pour l'entraînement
VAL_RATIO = 0.01          # Ratio de pixels pour la validation
MODEL_SIZE = "base"       # base / large / huge
NUM_CLASSES = 2           # 0=no-change, 1=change
IGNORE_INDEX = 255        # Valeur pour les pixels sans label
DEVICE = "cuda:0"

## 2. Imports et setup

In [ ]:
import sys, os, random
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
from sklearn.decomposition import PCA
from sklearn.metrics import (
    confusion_matrix, accuracy_score, classification_report,
    cohen_kappa_score, f1_score
)
from pathlib import Path

# Ajouter le dossier model au path pour les imports HyperSIGMA
sys.path.insert(0, os.path.abspath(MODEL_DIR))
from model import ss_fusion_seg

device = torch.device(DEVICE if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

def setup_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

setup_seed(42)

## 3. Chargement des données MUMUCD

On charge les images PRISMA hyperspectrales (230 bandes) avant et après, ainsi que le masque de changement binaire. Les deux cubes spectraux sont concaténés le long de l'axe spectral (→ 460 bandes) pour fournir au modèle l'information temporelle.

In [ ]:
data_dir = Path(DATA_ROOT) / CITY

# Charger PRISMA avant/après
prs_before = xr.open_dataset(data_dir / f"{CITY}-before-prs.nc")
prs_after  = xr.open_dataset(data_dir / f"{CITY}-after-prs.nc")

cube_before = prs_before['sr'].values.astype(np.float32)  # (H, W, 230)
cube_after  = prs_after['sr'].values.astype(np.float32)   # (H, W, 230)

print(f"PRISMA avant: {cube_before.shape}, après: {cube_after.shape}")

# Concaténer avant + après le long de l'axe spectral
data_raw = np.concatenate([cube_before, cube_after], axis=2)  # (H, W, 460)
print(f"Cube concaténé: {data_raw.shape}")

# Charger le masque de changement binaire : 0=no-change, 1=change
cd = xr.open_dataset(data_dir / f"{CITY}-cd-binary.nc")
data_gt = cd['change'].values.astype(np.int64)  # (H, W) : 0 ou 1

unique, counts = np.unique(data_gt, return_counts=True)
total = data_gt.size
print(f"\nDistribution des classes :")
for u, c in zip(unique, counts):
    label = {0: "Pas de changement", 1: "Changement"}[u]
    print(f"  Classe {u} ({label}): {c:,} pixels ({100*c/total:.2f}%)")

# Charger S2 pour la visualisation
s2_before = xr.open_dataset(data_dir / f"{CITY}-before-s2.nc")
s2_after  = xr.open_dataset(data_dir / f"{CITY}-after-s2.nc")

def s2_to_rgb(ds, r_idx=3, g_idx=2, b_idx=1, percentile_clip=98):
    rgb = ds['sr'].values[:, :, [r_idx, g_idx, b_idx]]
    vmin = np.nanpercentile(rgb, 2)
    vmax = np.nanpercentile(rgb, percentile_clip)
    return np.clip((rgb - vmin) / (vmax - vmin), 0, 1)

rgb_before = s2_to_rgb(s2_before)
rgb_after  = s2_to_rgb(s2_after)

prs_before.close(); prs_after.close(); cd.close()
s2_before.close(); s2_after.close()

## 4. Visualisation des données

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

axes[0].imshow(rgb_before)
axes[0].set_title(f"Avant (S2 RGB) — {CITY}")
axes[0].axis('off')

axes[1].imshow(rgb_after)
axes[1].set_title(f"Après (S2 RGB) — {CITY}")
axes[1].axis('off')

axes[2].imshow(rgb_after)
axes[2].imshow(change_mask, alpha=0.4, cmap='Reds')
axes[2].set_title("Après + masque de changement")
axes[2].axis('off')

plt.suptitle(f"Détection de changement — {CITY.replace('_', ' ').title()}", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Préparation des données (PCA + patches + split train/val/test)

In [ ]:
# ---- Remplacer NaN par 0 et normaliser ----
data_raw = np.nan_to_num(data_raw, nan=0.0)

# Normalisation min-max par bande
H, W, B = data_raw.shape
data_flat = data_raw.reshape(-1, B)
bmin = data_flat.min(axis=0, keepdims=True)
bmax = data_flat.max(axis=0, keepdims=True)
data_flat = (data_flat - bmin) / (bmax - bmin + 1e-8)
data_norm = data_flat.reshape(H, W, B)

print(f"Données normalisées: {data_norm.shape}, min={data_norm.min():.3f}, max={data_norm.max():.3f}")

# ---- PCA ----
print(f"Application PCA : {B} bandes → {PCA_COMPONENTS} composantes...")
data_2d = data_norm.reshape(-1, B)
pca = PCA(n_components=PCA_COMPONENTS, whiten=True)
data_pca_2d = pca.fit_transform(data_2d)
data = data_pca_2d.reshape(H, W, PCA_COMPONENTS).astype(np.float32)

print(f"Variance expliquée: {pca.explained_variance_ratio_.sum()*100:.1f}%")
print(f"Données après PCA: {data.shape}")

In [ ]:
# ---- Split train / val / test (par pixel) ----
gt_reshape = data_gt.reshape(-1)  # (H*W,) — values 0 or 1

# Tous les pixels sont étiquetés
all_idx = np.arange(len(gt_reshape))
np.random.shuffle(all_idx)

n_total = len(all_idx)
n_train = int(n_total * TRAIN_RATIO)
n_val   = int(n_total * VAL_RATIO)

train_index = all_idx[:n_train]
val_index   = all_idx[n_train:n_train + n_val]
test_index  = all_idx[n_train + n_val:]

print(f"Train: {len(train_index):,} pixels ({100*len(train_index)/n_total:.1f}%)")
print(f"Val:   {len(val_index):,} pixels ({100*len(val_index)/n_total:.1f}%)")
print(f"Test:  {len(test_index):,} pixels ({100*len(test_index)/n_total:.1f}%)")

# Créer les masques GT par split (IGNORE_INDEX = non utilisé dans ce split)
train_gt = np.full_like(gt_reshape, IGNORE_INDEX)
val_gt   = np.full_like(gt_reshape, IGNORE_INDEX)
test_gt  = np.full_like(gt_reshape, IGNORE_INDEX)

train_gt[train_index] = gt_reshape[train_index]
val_gt[val_index]     = gt_reshape[val_index]
test_gt[test_index]   = gt_reshape[test_index]

train_gt = train_gt.reshape(H, W)
val_gt   = val_gt.reshape(H, W)
test_gt  = test_gt.reshape(H, W)

# Vérifier la distribution dans le train set
for c in range(NUM_CLASSES):
    n = np.sum(train_gt == c)
    print(f"  Train classe {c}: {n:,} pixels")

In [ ]:
# ---- Extraction des patches ----
def get_patch(img_size, data, data_gt, overlap_size):
    """Extrait des patches avec overlap depuis le cube et le GT."""
    input_size = (img_size, img_size)
    height, width, bands = data.shape

    LyEnd, LxEnd = np.subtract((height, width), input_size)
    Lx = np.linspace(0, LxEnd, int(np.ceil(LxEnd / float(input_size[1] - overlap_size))) + 1,
                     endpoint=True).astype('int')
    Ly = np.linspace(0, LyEnd, int(np.ceil(LyEnd / float(input_size[0] - overlap_size))) + 1,
                     endpoint=True).astype('int')

    N = len(Ly) * len(Lx)
    X_data = np.zeros([N, input_size[0], input_size[1], bands], dtype=np.float32)
    y_data = np.zeros([N, input_size[0], input_size[1]], dtype=np.int64)

    i = 0
    for j in range(len(Ly)):
        for k in range(len(Lx)):
            rStart, cStart = Ly[j], Lx[k]
            rEnd, cEnd = rStart + input_size[0], cStart + input_size[1]
            X_data[i] = data[rStart:rEnd, cStart:cEnd, :]
            y_data[i] = data_gt[rStart:rEnd, cStart:cEnd]
            i += 1
    return X_data, y_data

img_train, gt_train_patches = get_patch(IMG_SIZE, data, train_gt, OVERLAP_SIZE)
img_val,   gt_val_patches   = get_patch(IMG_SIZE, data, val_gt,   OVERLAP_SIZE)
img_test,  gt_test_patches  = get_patch(IMG_SIZE, data, test_gt,  OVERLAP_SIZE)

print(f"Train patches: {img_train.shape}, labels: {gt_train_patches.shape}")
print(f"Val patches:   {img_val.shape},   labels: {gt_val_patches.shape}")
print(f"Test patches:  {img_test.shape},  labels: {gt_test_patches.shape}")

# Convertir labels en vecteurs pour le loss : (N, H*W)
gt_train_flat = gt_train_patches.reshape(img_train.shape[0], -1)
gt_val_flat   = gt_val_patches.reshape(img_val.shape[0], -1)
gt_test_flat  = gt_test_patches.reshape(img_test.shape[0], -1)

## 6. DataLoaders PyTorch

In [ ]:
class PatchDataset(Dataset):
    def __init__(self, img_patches, gt_patches):
        # img_patches: (N, H, W, C) → (N, C, H, W)
        self.x = torch.FloatTensor(img_patches.transpose(0, 3, 1, 2))
        self.y = torch.LongTensor(gt_patches)  # (N, H*W) — labels 0=ignoré, 1/2=classes

    def __len__(self):
        return self.x.shape[0]

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

train_ds = PatchDataset(img_train, gt_train_flat)
val_ds   = PatchDataset(img_val, gt_val_flat)
test_ds  = PatchDataset(img_test, gt_test_flat)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}")

## 7. Modèle HyperSIGMA + chargement des poids pré-entraînés

In [ ]:
model = ss_fusion_seg.SSFusionFramework(
    img_size=IMG_SIZE,
    in_channels=PCA_COMPONENTS,
    patch_size=PATCH_SIZE,
    classes=NUM_CLASSES,
    model_size=MODEL_SIZE
).to(device)

# ---- Charger les poids pré-entraînés (encodeurs spatial + spectral) ----
spat_path = os.path.join(WEIGHT_DIR, "spat-vit-base-ultra-checkpoint-1599.pth")
spec_path = os.path.join(WEIGHT_DIR, "spec-vit-base-ultra-checkpoint-1599.pth")

def load_pretrained_encoder(model, spat_path, spec_path):
    """Charge les poids pré-entraînés dans les encodeurs, en ignorant les clés incompatibles."""
    model_params = model.state_dict()
    loaded = 0

    for ckpt_path, prefix in [(spat_path, 'spat_encoder.'), (spec_path, 'spec_encoder.')]:
        if not os.path.exists(ckpt_path):
            print(f"  [SKIP] {ckpt_path} non trouvé — entraînement from scratch pour {prefix}")
            continue
        try:
            ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
            state = ckpt.get('model', ckpt)

            skip_patterns = ['patch_embed.proj', 'pos_embed', 'head', 'spat_map']
            for k in list(state.keys()):
                full_key = prefix + k
                if any(p in k for p in skip_patterns):
                    continue
                if full_key in model_params and state[k].shape == model_params[full_key].shape:
                    model_params[full_key] = state[k]
                    loaded += 1

            print(f"  [OK] {ckpt_path}: {loaded} paramètres chargés")
        except Exception as e:
            print(f"  [ERREUR] {ckpt_path}: {e} — entraînement from scratch pour {prefix}")

    model.load_state_dict(model_params)
    return model

model = load_pretrained_encoder(model, spat_path, spec_path)

# ---- Optimizer, scheduler, loss ----
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS + 1)

# ignore_index=IGNORE_INDEX pour ignorer les pixels sans label (255)
criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

n_params = sum(p.numel() for p in model.parameters()) / 1e6
n_train_params = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f"\nModèle: {n_params:.1f}M paramètres ({n_train_params:.1f}M entraînables)")

## 8. Entraînement (fine-tuning)

In [ ]:
os.makedirs("output", exist_ok=True)
path_weight = f"output/hypersigma_{CITY}_"

best_val_loss = float('inf')
train_losses, val_losses = [], []

for epoch in range(1, NUM_EPOCHS + 1):
    # ---- Train ----
    model.train()
    epoch_loss = 0
    n_batches = 0
    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)   # (B, C, H, W)
        batch_y = batch_y.to(device)   # (B, H*W)

        pred = model(batch_x)          # (B, H*W, NUM_CLASSES)
        pred = pred.reshape(-1, NUM_CLASSES)  # (B*H*W, NUM_CLASSES)
        target = batch_y.reshape(-1)          # (B*H*W,)

        loss = criterion(pred, target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        n_batches += 1

    scheduler.step()
    avg_train_loss = epoch_loss / max(n_batches, 1)
    train_losses.append(avg_train_loss)

    # ---- Validation (tous les 5 epochs) ----
    if epoch % 5 == 0 or epoch == 1:
        model.eval()
        val_loss = 0
        n_val = 0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x = batch_x.to(device)
                batch_y = batch_y.to(device)
                pred = model(batch_x)
                pred = pred.reshape(-1, NUM_CLASSES)
                target = batch_y.reshape(-1)
                loss = criterion(pred, target)
                val_loss += loss.item()
                n_val += 1

        avg_val_loss = val_loss / max(n_val, 1)
        val_losses.append(avg_val_loss)

        print(f"Epoch {epoch:3d}/{NUM_EPOCHS} — train_loss: {avg_train_loss:.4f}, val_loss: {avg_val_loss:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), path_weight + "best.pt")
            print(f"  → Meilleur modèle sauvegardé (val_loss={best_val_loss:.4f})")
    else:
        val_losses.append(val_losses[-1] if val_losses else avg_train_loss)

print(f"\nEntraînement terminé. Meilleur val_loss: {best_val_loss:.4f}")

In [ ]:
# Courbes de loss
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_losses, label='Train loss')
ax.plot(val_losses, label='Val loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title(f'Courbes de loss — {CITY}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Inférence sur l'image complète

In [ ]:
torch.cuda.empty_cache()

# Charger le meilleur modèle
model.load_state_dict(torch.load(path_weight + "best.pt", map_location=device, weights_only=True))
model.eval()

# Calculer la grille de patches pour l'inférence
input_size = (IMG_SIZE, IMG_SIZE)
LyEnd, LxEnd = np.subtract((H, W), input_size)
Lx = np.linspace(0, LxEnd, int(np.ceil(LxEnd / float(input_size[1] - OVERLAP_SIZE))) + 1,
                 endpoint=True).astype('int')
Ly = np.linspace(0, LyEnd, int(np.ceil(LyEnd / float(input_size[0] - OVERLAP_SIZE))) + 1,
                 endpoint=True).astype('int')

# Prédiction par patch avec fusion par moyenne des logits
pred_sum   = np.zeros((H, W, NUM_CLASSES), dtype=np.float32)
pred_count = np.zeros((H, W, 1), dtype=np.float32)

print(f"Inférence : {len(Ly)}x{len(Lx)} = {len(Ly)*len(Lx)} patches")

with torch.no_grad():
    for j in range(len(Ly)):
        for k in range(len(Lx)):
            rStart, cStart = Ly[j], Lx[k]
            rEnd, cEnd = rStart + IMG_SIZE, cStart + IMG_SIZE

            patch = data[rStart:rEnd, cStart:cEnd, :]
            patch_t = torch.FloatTensor(patch).permute(2, 0, 1).unsqueeze(0).to(device)

            out = model(patch_t)  # (1, H*W, NUM_CLASSES)
            out = out.reshape(IMG_SIZE, IMG_SIZE, NUM_CLASSES).cpu().numpy()

            pred_sum[rStart:rEnd, cStart:cEnd] += out
            pred_count[rStart:rEnd, cStart:cEnd] += 1

# Moyenne des prédictions dans les zones de chevauchement
pred_avg = pred_sum / np.maximum(pred_count, 1)
y_pred_full = np.argmax(pred_avg, axis=2)  # (H, W) — 0=no-change, 1=change

print(f"Prédiction complète: {y_pred_full.shape}")
print(f"Valeurs uniques prédites: {np.unique(y_pred_full, return_counts=True)}")

## 10. Évaluation sur le test set

In [ ]:
# Évaluation uniquement sur les pixels du test set
test_gt_flat = test_gt.reshape(-1)
y_pred_flat  = y_pred_full.reshape(-1)

# Masque des pixels de test (non IGNORE_INDEX)
test_mask = test_gt_flat != IGNORE_INDEX
y_true_test = test_gt_flat[test_mask]
y_pred_test = y_pred_flat[test_mask]

# Métriques
oa = accuracy_score(y_true_test, y_pred_test)
kappa = cohen_kappa_score(y_true_test, y_pred_test)
f1 = f1_score(y_true_test, y_pred_test, average='binary', pos_label=1)

print(f"=== Résultats sur le test set ({len(y_true_test):,} pixels) ===")
print(f"  Overall Accuracy (OA): {oa*100:.2f}%")
print(f"  Kappa:                 {kappa:.4f}")
print(f"  F1-score (change):     {f1*100:.2f}%")
print()
print(classification_report(y_true_test, y_pred_test, target_names=["No-change", "Change"]))

# Matrice de confusion
cm = confusion_matrix(y_true_test, y_pred_test)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i,j]:,}", ha='center', va='center',
                color='white' if cm[i,j] > cm.max()/2 else 'black', fontsize=12)
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(["No-change", "Change"])
ax.set_yticklabels(["No-change", "Change"])
ax.set_xlabel("Prédit"); ax.set_ylabel("Vérité terrain")
ax.set_title(f"Matrice de confusion — {CITY} (OA={oa*100:.1f}%, F1={f1*100:.1f}%)")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## 11. Visualisation des résultats

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(22, 14))

# Ligne 1 : images S2 + vérité terrain
axes[0, 0].imshow(rgb_before)
axes[0, 0].set_title(f"Avant (S2 RGB) — {CITY}")
axes[0, 0].axis('off')

axes[0, 1].imshow(rgb_after)
axes[0, 1].set_title(f"Après (S2 RGB) — {CITY}")
axes[0, 1].axis('off')

axes[0, 2].imshow(data_gt, cmap='RdYlGn_r', vmin=0, vmax=1)
axes[0, 2].set_title("Vérité terrain (changement)")
axes[0, 2].axis('off')

# Ligne 2 : prédiction + overlay + erreurs
axes[1, 0].imshow(y_pred_full, cmap='RdYlGn_r', vmin=0, vmax=1)
axes[1, 0].set_title("Prédiction HyperSIGMA")
axes[1, 0].axis('off')

axes[1, 1].imshow(rgb_after)
axes[1, 1].imshow(y_pred_full, alpha=0.4, cmap='Reds')
axes[1, 1].set_title("Après + prédiction overlay")
axes[1, 1].axis('off')

# Carte d'erreurs (uniquement sur les pixels de test)
error_map = np.full((H, W), np.nan)
test_mask_2d = test_gt != IGNORE_INDEX
error_map[test_mask_2d] = (y_pred_full[test_mask_2d] != data_gt[test_mask_2d]).astype(float)

axes[1, 2].imshow(rgb_after, alpha=0.3)
im_err = axes[1, 2].imshow(error_map, cmap='RdYlGn_r', vmin=0, vmax=1, alpha=0.7)
axes[1, 2].set_title("Erreurs de classification (rouge = erreur)")
axes[1, 2].axis('off')

plt.suptitle(f"HyperSIGMA Change Detection — {CITY.replace('_', ' ').title()}\n"
             f"OA={oa*100:.1f}%, F1={f1*100:.1f}%, Kappa={kappa:.3f}", fontsize=14)
plt.tight_layout()
plt.show()

## 12. Zoom sur la zone de changement la plus dense

In [ ]:
# Trouver la zone avec la plus forte densité de changement (fenêtre 256x256)
kernel_size = 256
cm_float = data_gt.astype(float)
cumsum = np.cumsum(np.cumsum(cm_float, axis=0), axis=1)
density = np.zeros_like(cm_float)
k = kernel_size
density[k:, k:] = (cumsum[k:, k:] - cumsum[:-k, k:] - cumsum[k:, :-k] + cumsum[:-k, :-k]) / (k * k)

cy, cx = np.unravel_index(np.argmax(density), density.shape)
half = kernel_size // 2
r_start, r_end = max(0, cy - half), min(H, cy + half)
c_start, c_end = max(0, cx - half), min(W, cx + half)
s = (slice(r_start, r_end), slice(c_start, c_end))

fig, axes = plt.subplots(2, 4, figsize=(24, 12))

# Ligne 1 : vues globales avec rectangle
for ax, img, title in [
    (axes[0,0], rgb_before, "Avant (S2)"),
    (axes[0,1], rgb_after, "Après (S2)"),
]:
    ax.imshow(img)
    rect = plt.Rectangle((c_start, r_start), c_end-c_start, r_end-r_start,
                          linewidth=2, edgecolor='red', facecolor='none')
    ax.add_patch(rect)
    ax.set_title(title); ax.axis('off')

axes[0,2].imshow(data_gt, cmap='Reds')
rect = plt.Rectangle((c_start, r_start), c_end-c_start, r_end-r_start,
                      linewidth=2, edgecolor='blue', facecolor='none')
axes[0,2].add_patch(rect)
axes[0,2].set_title("GT changement"); axes[0,2].axis('off')

axes[0,3].imshow(y_pred_full, cmap='Reds')
rect = plt.Rectangle((c_start, r_start), c_end-c_start, r_end-r_start,
                      linewidth=2, edgecolor='blue', facecolor='none')
axes[0,3].add_patch(rect)
axes[0,3].set_title("Prédiction"); axes[0,3].axis('off')

# Ligne 2 : zoom
axes[1,0].imshow(rgb_before[s[0], s[1]])
axes[1,0].set_title("Avant (zoom)"); axes[1,0].axis('off')

axes[1,1].imshow(rgb_after[s[0], s[1]])
axes[1,1].set_title("Après (zoom)"); axes[1,1].axis('off')

axes[1,2].imshow(data_gt[s[0], s[1]], cmap='Reds', vmin=0, vmax=1)
axes[1,2].set_title("GT (zoom)"); axes[1,2].axis('off')

axes[1,3].imshow(y_pred_full[s[0], s[1]], cmap='Reds', vmin=0, vmax=1)
axes[1,3].set_title("Prédiction (zoom)"); axes[1,3].axis('off')

plt.suptitle(f"Zoom — zone de changement la plus dense ({CITY})", fontsize=14)
plt.tight_layout()
plt.show()